# 02 · Factorial de dos factores — ANOVA de dos vías (R)

**Semana 2 — Bloqueo y factoriales.**

## Objetivos
- Ajustar un factorial de dos factores con interacción.
- Interpretar la **gráfica de interacción** y la estrategia de análisis.

> Teoría: [`../teoria/02-factoriales-interacciones.md`](../teoria/02-factoriales-interacciones.md) y [`../teoria/03-anova-dos-vias.md`](../teoria/03-anova-dos-vias.md).

In [ ]:
suppressMessages(library(car))
set.seed(2)

## 1. Los datos

Vida (h) de una batería según **material** (3) y **temperatura** (15/70/125 °F), $n=4$ (Montgomery, ej. 5.3).

In [ ]:
df <- read.csv('../datos/vida-bateria.csv')
df$material <- factor(df$material)
df$temperatura <- factor(df$temperatura)
with(df, tapply(vida, list(material, temperatura), mean))

## 2. Gráfica de interacción

In [ ]:
with(df, interaction.plot(temperatura, material, vida,
     xlab='Temperatura (°F)', ylab='Vida media (h)',
     main='Interacción material × temperatura', lwd=2, col=1:3))

Las líneas **no** son paralelas: se anticipa una **interacción**.

## 3. ANOVA de dos vías con interacción

In [ ]:
modelo <- aov(vida ~ material * temperatura, data = df)
summary(modelo)

**Mirar primero la interacción:** `material:temperatura` es significativa ($p\approx0.019$). Material y temperatura actúan conjuntamente; los efectos principales se interpretan con cautela.

## 4. Verificación de supuestos

In [ ]:
op <- par(mfrow = c(1, 2))
qqnorm(residuals(modelo)); qqline(residuals(modelo))
plot(fitted(modelo), residuals(modelo), xlab='Ajustados', ylab='Residuales',
     main='Residuales vs. ajustados'); abline(h=0, lty=2)
par(op)
shapiro.test(residuals(modelo))

## 5. Conclusiones
- **Interacción significativa** material × temperatura.
- A alta temperatura el **material 3** es superior.

> Equivalente en Python: [`02-factorial-dos-vias_py.ipynb`](02-factorial-dos-vias_py.ipynb).

---

## Ejemplo 2 — Cuando la interacción NO es significativa (modelo aditivo)

**Rendimiento** (kg/parcela) de **tres variedades** de trigo (V1–V3) bajo **dos niveles de riego** (Bajo, Alto), $n=3$. ¿Qué cambia cuando los factores **no** interactúan?

In [ ]:
fa <- expand.grid(rep = 1:3, riego = c('Bajo', 'Alto'),
                  variedad = c('V1', 'V2', 'V3'))
fa$rend <- c(19,20,21, 23,24,25,    # V1
             24,25,26, 28,29,30,    # V2
             21,22,23, 25,26,27)    # V3
fa$variedad <- factor(fa$variedad); fa$riego <- factor(fa$riego, c('Bajo', 'Alto'))
with(fa, tapply(rend, list(variedad, riego), mean))

In [ ]:
with(fa, interaction.plot(riego, variedad, rend,
     xlab = 'Riego', ylab = 'Rendimiento medio (kg/parcela)',
     main = 'Interacción variedad × riego', lwd = 2, col = 1:3))

Líneas **paralelas**: el salto Bajo→Alto (+4 kg) es igual en las tres variedades. No se anticipa interacción.

In [ ]:
mod_t <- aov(rend ~ variedad * riego, data = fa)
summary(mod_t)

La interacción **no** es significativa ($p\approx1.0$). Como el modelo es **aditivo**, reajustamos **sin** interacción e interpretamos los efectos principales con Tukey.

In [ ]:
mod_ta <- aov(rend ~ variedad + riego, data = fa)
summary(mod_ta)
TukeyHSD(mod_ta, 'variedad')

Ambos efectos principales significativos. Tukey: las tres variedades difieren y **V2** (27 kg) rinde más. El riego Alto suma $\approx4$ kg **por igual** en todas (sin interacción). Recomendación: **V2 con riego Alto**.

---

## Ejemplo 3 — Interacción de cruce: cuando el efecto principal engaña

Dos **métodos de enseñanza** (Tradicional, Activo) en estudiantes de **nivel previo** Bajo/Alto; respuesta: **puntaje** final, $n=3$.

In [ ]:
fc <- expand.grid(rep = 1:3, nivel = c('Bajo', 'Alto'),
                  metodo = c('Tradicional', 'Activo'))
fc$puntaje <- c(53,55,57, 86,88,90,    # Tradicional
                78,80,82, 68,70,72)    # Activo
fc$metodo <- factor(fc$metodo); fc$nivel <- factor(fc$nivel, c('Bajo', 'Alto'))
with(fc, tapply(puntaje, list(metodo, nivel), mean))

In [ ]:
with(fc, interaction.plot(nivel, metodo, puntaje,
     xlab = 'Nivel previo', ylab = 'Puntaje medio',
     main = 'Interacción método × nivel (cruce)', lwd = 2, col = 1:2))

In [ ]:
mod_e <- aov(puntaje ~ metodo * nivel, data = fc)
summary(mod_e)
cat('\nMedia por método (promediando nivel):\n')
print(tapply(fc$puntaje, fc$metodo, mean))

Interacción **enorme** ($F\approx347$, $p<10^{-6}$). El efecto principal del método (Activo 75.0 vs. Tradicional 71.5) sugiere "casi lo mismo" — **engañoso**: el método Activo ayuda al nivel Bajo (80 vs. 55) pero **perjudica** al Alto (70 vs. 88). Las líneas se **cruzan**. Recomendación **condicionada**: Activo para nivel Bajo, Tradicional para nivel Alto.

---

## Ejemplo 4 — Un $2\times2$ a mano: efectos principales e interacción

**Temperatura** (bajo/alto) y **catalizador** (bajo/alto) sobre el **rendimiento** (%), $n=3$. Calculamos los efectos con las fórmulas de la teoría.

In [ ]:
f4 <- expand.grid(rep = 1:3, cat = c('bajo', 'alto'), temp = c('bajo', 'alto'))
f4$rend <- c(18,20,22, 23,25,27,    # temp bajo: cat bajo, cat alto
             28,30,32, 50,52,54)    # temp alto: cat bajo, cat alto
f4$temp <- factor(f4$temp); f4$cat <- factor(f4$cat)
m <- with(f4, tapply(rend, list(temp, cat), mean))
efecto_temp <- ((m['alto','bajo'] + m['alto','alto']) - (m['bajo','bajo'] + m['bajo','alto'])) / 2
efecto_cat  <- ((m['bajo','alto'] + m['alto','alto']) - (m['bajo','bajo'] + m['alto','bajo'])) / 2
efecto_int  <- ((m['alto','alto'] - m['bajo','alto']) - (m['alto','bajo'] - m['bajo','bajo'])) / 2
cat('Efecto temperatura =', efecto_temp, '\n')
cat('Efecto catalizador =', efecto_cat, '\n')
cat('Efecto interacción =', efecto_int, '\n')

Efectos: temperatura $=18.5$, catalizador $=13.5$, interacción $=8.5$. La interacción positiva indica que ambos factores **se potencian** (la celda alto–alto se dispara). Confirmamos con el ANOVA:

In [ ]:
mod_q <- aov(rend ~ temp * cat, data = f4)
summary(mod_q)

Los tres términos son significativos; la interacción ($F\approx54$, $p\approx0.0001$) confirma el cálculo a mano. En un $2^k$ (Semana 3) esto se generaliza a muchos factores.